# SpectraShift Week 2 completion from verified staging
Attach the source dataset and the saved output from the Version 3 run. Use a T4 GPU. This notebook does not download or copy the 63 GB archive or the 18.46 GB staged arrays.

In [ ]:
from pathlib import Path
import json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/spectrashift-week2-completion')
WORK.mkdir(parents=True, exist_ok=True)
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift').is_dir()]
if not projects:
    source_bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(source_bundles) == 1, f'Expected one SpectraShift source bundle, found {source_bundles}'
    extracted_source = WORK / 'source'
    shutil.unpack_archive(str(source_bundles[0]), str(extracted_source))
    projects = [extracted_source]
staging_summaries = sorted(INPUT.rglob('staging_summary.json'))
partition_manifests = sorted(INPUT.rglob('partitions.parquet'))
freeze_summaries = sorted(INPUT.rglob('freeze_summary.json'))
assert len(projects) == 1, f'Expected one SpectraShift source tree, found {projects}'
assert len(staging_summaries) == 1, f'Expected one saved staging output, found {staging_summaries}'
assert len(partition_manifests) == 1, f'Expected one frozen manifest, found {partition_manifests}'
assert len(freeze_summaries) == 1, f'Expected one freeze summary, found {freeze_summaries}'
PROJECT = projects[0]
STAGED = staging_summaries[0].parent
MANIFEST = partition_manifests[0].parent
FREEZE_SUMMARY = freeze_summaries[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)
print({'project': str(PROJECT), 'staged': str(STAGED), 'manifest': str(MANIFEST)})


In [ ]:
config = yaml.safe_load((PROJECT / 'configs/data/week2.yaml').read_text())
config['staging']['output_dir'] = str(STAGED)
config['staging']['final_manifest_dir'] = str(MANIFEST)
config['staging']['report_dir'] = str(WORK / 'reports')
runtime_config = WORK / 'week2-completion.yaml'
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
staging = json.loads(staging_summaries[0].read_text())
freeze = json.loads(FREEZE_SUMMARY.read_text())
normalization = json.loads((MANIFEST / 'normalization.json').read_text())
assert staging['incomplete_patches'] == 0 and staging['complete_patches'] == 50200, staging
assert staging['archive_part_sha256']['BigEarthNet-S2.tar.gzaa'] == 'a44e4c9d8dde1affd7107640dd0070b811ca848205363f8dd8f75bebccea0ed3', staging
assert staging['archive_part_sha256']['BigEarthNet-S2.tar.gzab'] == 'e0bfe57038e07ff09add42e5bff108d221da10a6e45533fd2eb4ac05e6d90dc2', staging
assert freeze['training_approved'] and freeze['evaluation_block_gate'] and freeze['support_gate'], freeze
print(json.dumps({'staging_verified': True, 'freeze': freeze, 'normalization_sha256': normalization['sha256']}, indent=2))


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU accelerator before running'
major, minor = torch.cuda.get_device_capability(0)
device_arch = f'sm_{major}{minor}'
supported_arches = torch.cuda.get_arch_list()
print({'gpu': torch.cuda.get_device_name(0), 'device_arch': device_arch, 'pytorch_arches': supported_arches})
assert device_arch in supported_arches, f'{torch.cuda.get_device_name(0)} ({device_arch}) is unsupported by this PyTorch build; select a T4 GPU'


In [ ]:
from spectrashift.train.throughput import benchmark_throughput
throughput = benchmark_throughput(runtime_config)
print(json.dumps(throughput, indent=2))


In [ ]:
from spectrashift.data.smoke import run_smoke
smoke = run_smoke(runtime_config)
assert smoke['overfit_gate'], smoke
print(json.dumps(smoke, indent=2))


In [ ]:
summary = {'staging': staging, 'freeze': freeze, 'normalization_sha256': normalization['sha256'], 'throughput': throughput, 'smoke': smoke}
summary_path = WORK / 'week2_run_summary.json'
summary_path.write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps({'week2_complete': True, 'summary_path': str(summary_path)}, indent=2))
